# Spectral Extraction & NDVI Vegetation Validation

This notebook extracts hyperspectral reflectance values from AVIRIS-NG tiles at field-collected species locations, then filters out any points that do not represent real vegetation using an NDVI threshold.

**This is to be used on an R/RStudio, not Python**

## 1. Load Libraries

In [ ]:
library(terra)
library(sf)
library(dplyr)
library(ncdf4)
library(ggplot2)

## 2. Configuration

All file paths and thresholds are defined here. Update the directory to where your files are stored.

The NDVI thresholds control what counts as a valid vegetation pixel:
- **NDVI < 0.2** → bare soil, urban surfaces, or water — not vegetation
- **NDVI > 0.95** → likely cloud shadow or a sensor error

In [ ]:
# File paths
SPECIES_CSV <- "Data/Hyperspectral/Outputs/Species_data_final4.csv"
AVIRIS_DIR  <- "Data/TMNP_AVIRIS_L3"
OUT_RDS     <- "Data/Hyperspectral/Outputs/species_spectra4.rds"
OUT_CSV     <- "Data/Hyperspectral/Outputs/species_spectra4.csv"
FIG_DIR     <- "Data/Hyperspectral/Outputs/Figures"
dir.create(FIG_DIR, recursive=TRUE, showWarnings=FALSE)

# NDVI thresholds
NDVI_MIN <- 0.2
NDVI_MAX <- 0.95

The crs below is the coordinate reference system the AVIRIS tiles are stored in. We need to reproject our GPS species points into this system before we can extract pixel values from the tiles.

In [ ]:
# AVIRIS-NG Albers Equal Area projection (South Africa)
AVIRIS_CRS <- paste0(
  'PROJCS["unnamed",GEOGCS["Ellipse Based",',
  'DATUM["Ellipse Based",SPHEROID["Unnamed",6378137,298.257223562997]],',
  'PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433]],',
  'PROJECTION["Albers_Conic_Equal_Area"],',
  'PARAMETER["latitude_of_center",-30],',
  'PARAMETER["longitude_of_center",25],',
  'PARAMETER["standard_parallel_1",-22],',
  'PARAMETER["standard_parallel_2",-38],',
  'PARAMETER["false_easting",1400000],',
  'PARAMETER["false_northing",1300000],',
  'UNIT["metre",1]]'
)

## 3. Load Species Data

We load the field-collected species records, which contain GPS coordinates (latitude/longitude) and species names. We then convert these points from standard GPS coordinates (WGS84) into the AVIRIS tile projection (Albers) so they can be overlaid on the imagery.

In [ ]:
species_clean <- read.csv(SPECIES_CSV, stringsAsFactors=FALSE)

cat("Species records loaded:", nrow(species_clean), "\n")
cat("\nSamples per species:\n")
print(table(species_clean$Scientific.Names))

Convert the data frame to a spatial object using Longitude and Latitude columns.

`crs = 4326` means standard GPS coordinates (WGS84).

`remove = FALSE` keeps the original coordinate columns in the table.

In [ ]:
species_sf   <- sf::st_as_sf(species_clean, coords=c("Longitude", "Latitude"),
                               crs=4326, remove=FALSE)

# Reproject the points from GPS (WGS84) into the AVIRIS Albers projection
species_proj <- sf::st_transform(species_sf, AVIRIS_CRS)

## 4. Read Wavelengths from the First AVIRIS Tile

All AVIRIS-NG tiles share the same wavelength axis, so we only need to read it once from the first tile. Each band's centre wavelength (in nanometres) will become a column name in our output spectral library — for example `B1_402nm`, `B2_412nm`, and so on.

In [ ]:
aviris_files <- sort(list.files(AVIRIS_DIR, pattern="AVIRIS.*\\.nc$", full.names=TRUE))
cat("AVIRIS tiles found:", length(aviris_files), "\n")

In [ ]:
# Open the first tile, read the wavelength variable, then close the file
nc_tmp         <- ncdf4::nc_open(aviris_files[1])
wavelengths_nm <- as.numeric(ncdf4::ncvar_get(nc_tmp, "reflectance/wavelength"))
ncdf4::nc_close(nc_tmp)

if (is.null(wavelengths_nm)) stop("Could not read wavelengths from NetCDF.")

cat("Wavelengths:", length(wavelengths_nm), "bands,",
    round(min(wavelengths_nm)), "–", round(max(wavelengths_nm)), "nm\n")

## 5. Extract Spectra Tile by Tile

We loop through every AVIRIS tile. For each tile we:
1. Read its spatial extent from the NetCDF coordinate dimensions
2. Check whether any of our species points fall inside it — if not, we skip it
3. Load the reflectance raster and sample the pixel value at each species point

Processing one tile at a time avoids loading everything into memory at once.

In [ ]:
all_extractions <- lapply(aviris_files, function(f) {

  # Read the easting and northing coordinate axes stored in the tile
  nc_tmp   <- ncdf4::nc_open(f)
  easting  <- as.numeric(nc_tmp$dim[["easting"]]$vals)
  northing <- as.numeric(nc_tmp$dim[["northing"]]$vals)
  ncdf4::nc_close(nc_tmp)

  # Calculate pixel size and build the tile's bounding box,
  # adding half a pixel on each side so the edges align correctly
  xres <- mean(diff(easting))
  yres <- mean(diff(northing))
  true_ext <- terra::ext(
    min(easting)  - abs(xres)/2,  max(easting)  + abs(xres)/2,
    min(northing) - abs(yres)/2,  max(northing) + abs(yres)/2
  )

  # Build an sf bounding box so we can check point overlap
  tile_bbox <- sf::st_bbox(c(
    xmin = min(easting)  - abs(xres)/2,
    xmax = max(easting)  + abs(xres)/2,
    ymin = min(northing) - abs(yres)/2,
    ymax = max(northing) + abs(yres)/2
  ), crs = sf::st_crs(AVIRIS_CRS))

  # Find which species points fall inside this tile
  pts_in_tile <- tryCatch(
    sf::st_crop(species_proj, tile_bbox),
    error = function(e) species_proj[0, ]  # return empty if crop fails
  )

  # Skip this tile entirely if no species points fall within it
  if (nrow(pts_in_tile) == 0) {
    cat("  Skipping (no points):", basename(f), "\n")
    return(NULL)
  }

  cat("  Extracting:", basename(f), "—", nrow(pts_in_tile), "points\n")

  # Load the reflectance raster, set its CRS and correct spatial extent,
  # then name each band using its wavelength (e.g. B1_402nm)
  r <- terra::rast(f, subds="/reflectance/reflectance")
  terra::crs(r) <- AVIRIS_CRS
  terra::ext(r) <- true_ext
  names(r)      <- paste0("B", seq_along(wavelengths_nm), "_",
                           round(wavelengths_nm), "nm")

  # Extract the pixel values at each species point location
  vals   <- terra::extract(r, terra::vect(pts_in_tile))

  # Combine species metadata with the extracted band values
  result <- cbind(
    sf::st_drop_geometry(pts_in_tile),  # species name, coordinates, etc.
    vals[, -1]                           # band values (drop the ID column)
  )
})

# Remove NULL entries (tiles with no points) and combine into one table
all_extractions <- Filter(Negate(is.null), all_extractions)
species_spectra <- do.call(rbind, all_extractions)
rownames(species_spectra) <- NULL

cat("\nRaw extractions:", nrow(species_spectra), "points\n")
cat("\nPer species:\n")
print(table(species_spectra$Scientific.Names))

## 6. Identify Band Columns and Check Reflectance Scaling

AVIRIS data is sometimes stored as integers multiplied by 10,000 to save space (e.g. a reflectance of 0.05 is stored as 500). We check a sample of values and divide by 10,000 if needed, so all reflectance values end up in the 0–1 range.

In [ ]:
# Identify which columns contain spectral band data
band_cols <- grep("^B[0-9]+_[0-9]+nm$", names(species_spectra), value=TRUE)
cat("Band columns:", length(band_cols), "\n")

# Sample a few mid-range band values to check the scale
sample_vals <- unlist(species_spectra[1, band_cols[50:55]])
cat("Sample reflectance values:", round(sample_vals, 4), "\n")

if (any(!is.na(sample_vals)) && max(sample_vals, na.rm=TRUE) > 2) {
  cat("Values appear to be ×10000 — dividing by 10000\n")
  species_spectra[, band_cols] <- species_spectra[, band_cols] / 10000
} else {
  cat("Values appear to already be in the 0–1 range ✓\n")
}

## 7. Calculate NDVI

NDVI (Normalised Difference Vegetation Index) uses the red and near-infrared bands to measure how strongly a pixel represents vegetation. Healthy vegetation absorbs red light and reflects NIR strongly, giving high NDVI. Non-vegetation surfaces (soil, buildings, water) give low or negative values.

$$NDVI = \frac{NIR - Red}{NIR + Red}$$

We find the bands closest to Red (670 nm) and NIR (800 nm) in our wavelength array.

In [ ]:
# Find the band index closest to Red (670 nm) and NIR (800 nm)
idx_red <- which.min(abs(wavelengths_nm - 670))
idx_nir <- which.min(abs(wavelengths_nm - 800))
col_red <- band_cols[idx_red]
col_nir <- band_cols[idx_nir]

cat(sprintf("Red band : %s (%.0f nm)\n", col_red, wavelengths_nm[idx_red]))
cat(sprintf("NIR band : %s (%.0f nm)\n", col_nir, wavelengths_nm[idx_nir]))

In [ ]:
B_red <- as.numeric(species_spectra[[col_red]])
B_nir <- as.numeric(species_spectra[[col_nir]])

# Small epsilon (1e-9) prevents division by zero
species_spectra$ndvi <- (B_nir - B_red) / (B_nir + B_red + 1e-9)

In [ ]:
ndvi_summary <- species_spectra %>%
  dplyr::group_by(Scientific.Names) %>%
  dplyr::summarise(
    n         = dplyr::n(),
    ndvi_mean = round(mean(ndvi, na.rm=TRUE), 3),
    ndvi_min  = round(min(ndvi,  na.rm=TRUE), 3),
    ndvi_max  = round(max(ndvi,  na.rm=TRUE), 3),
    n_low     = sum(ndvi < NDVI_MIN, na.rm=TRUE),
    .groups   = "drop"
  ) %>%
  dplyr::arrange(ndvi_mean)

print(ndvi_summary, n=Inf)

## 8. Plot NDVI Distribution Before Filtering

This boxplot shows the spread of NDVI values for each species before any filtering. Points to the left of the red dashed line will be removed — they are not registering as vegetation in the sensor data.

In [ ]:
p_ndvi <- ggplot(species_spectra,
                  aes(x=ndvi,
                      y=reorder(Scientific.Names, ndvi, FUN=median),
                      fill=Scientific.Names)) +
  geom_boxplot(alpha=0.75, outlier.size=0.8, show.legend=FALSE) +
  geom_vline(xintercept=NDVI_MIN, colour="red", linetype="dashed", linewidth=0.9) +
  annotate("text", x=NDVI_MIN+0.01, y=0.5,
           label=paste0("Min threshold\n(", NDVI_MIN, ")"),
           hjust=0, vjust=0, colour="red", size=3.2) +
  scale_x_continuous(limits=c(-0.1, 1.0)) +
  labs(
    title    = "NDVI Distribution by Species — Before Vegetation Filter",
    subtitle = "Points left of the red dashed line will be removed as non-vegetation",
    x        = "NDVI",
    y        = NULL
  ) +
  theme_bw(base_size=11) +
  theme(
    axis.text.y      = element_text(face="italic", size=9),
    panel.grid.minor = element_blank()
  )

In [ ]:
# Print
p_ndvi

In [ ]:
ggsave(file.path(FIG_DIR, "QC_NDVI_before_filter.png"),
       p_ndvi, width=12, height=9, dpi=300, bg="white")

## 9. Apply NDVI Filter and Remove Incomplete Spectra

We apply two filters:
1. **NDVI range filter** — removes points below 0.2 (not vegetation) and above 0.95 (cloud shadow or sensor error)
2. **Missing band filter** — removes any spectrum where more than 50% of bands are missing (NA), as these cannot be used in classification

In [ ]:
n_before <- nrow(species_spectra)

# Filter 1: keep only points with NDVI in the valid vegetation range
species_spectra <- species_spectra[
  !is.na(species_spectra$ndvi) &
  species_spectra$ndvi >= NDVI_MIN &
  species_spectra$ndvi <= NDVI_MAX, ]

cat(sprintf("NDVI filter: %d → %d points (%d removed)\n",
            n_before, nrow(species_spectra), n_before - nrow(species_spectra)))

In [ ]:
# Filter 2: remove rows where more than 50% of band values are missing
n_after_ndvi <- nrow(species_spectra)
frac_na      <- rowMeans(is.na(species_spectra[, band_cols]))
species_spectra <- species_spectra[frac_na <= 0.5, ]

cat(sprintf("Missing band filter: %d → %d points (%d removed)\n",
            n_after_ndvi, nrow(species_spectra), n_after_ndvi - nrow(species_spectra)))

In [ ]:
final_counts <- sort(table(species_spectra$Scientific.Names), decreasing=TRUE)
print(final_counts)

In [ ]:
# Warn if any species has very few points remaining
low_n <- final_counts[final_counts < 5]
if (length(low_n) > 0) {
  cat("\n⚠ Species with fewer than 5 points — consider removing from analysis:\n")
  print(low_n)
}

## 10. Plot NDVI Distribution After Filtering

This confirms that all remaining spectra have a valid vegetation NDVI signal.

In [ ]:
p_ndvi_after <- ggplot(species_spectra,
                        aes(x=ndvi,
                            y=reorder(Scientific.Names, ndvi, FUN=median),
                            fill=Scientific.Names)) +
  geom_boxplot(alpha=0.75, outlier.size=0.8, show.legend=FALSE) +
  scale_x_continuous(limits=c(0, 1.0)) +
  labs(
    title    = "NDVI Distribution by Species — After Vegetation Filter",
    subtitle = paste0("All points have NDVI ≥ ", NDVI_MIN, " confirming vegetation signal"),
    x        = "NDVI",
    y        = NULL
  ) +
  theme_bw(base_size=11) +
  theme(
    axis.text.y      = element_text(face="italic", size=9),
    panel.grid.minor = element_blank()
  )

In [ ]:
# Print
p_ndvi_after

In [ ]:
ggsave(file.path(FIG_DIR, "QC_NDVI_after_filter.png"),
       p_ndvi_after, width=12, height=9, dpi=300, bg="white")

## 11. NIR Sanity Check

A final check: in healthy vegetation, near-infrared (NIR) reflectance should always be higher than red reflectance. If fewer than 80% of species show this pattern, it suggests some non-vegetation contamination remains in the dataset and the NDVI threshold should be raised.

In [ ]:
nir_check <- species_spectra %>%
  dplyr::group_by(Scientific.Names) %>%
  dplyr::summarise(
    mean_red   = round(mean(.data[[col_red]], na.rm=TRUE), 4),
    mean_nir   = round(mean(.data[[col_nir]], na.rm=TRUE), 4),
    nir_gt_red = round(mean(.data[[col_nir]] > .data[[col_red]], na.rm=TRUE), 3),
    .groups    = "drop"
  )
print(nir_check)

In [ ]:
pct_nir_ok <- mean(nir_check$nir_gt_red) * 100
cat(sprintf("\n%.0f%% of species have mean NIR > mean Red ✓\n", pct_nir_ok))

if (pct_nir_ok < 80) {
  cat("  WARNING: Less than 80% of species show expected NIR > Red.\n")
  cat("  Consider raising NDVI_MIN to 0.3 or higher.\n")
}

## 12. Save the Final Spectral Library

The cleaned and validated spectral library is saved as both an RDS file (for fast reloading in R) and a CSV file (for the Python classification pipeline).

In [ ]:
saveRDS(species_spectra, OUT_RDS)
write.csv(species_spectra, OUT_CSV, row.names=FALSE)

cat("\n", strrep("=", 55), "\n")
cat("Extraction + validation complete ✓\n")
cat(sprintf("  Final dataset : %d points, %d species\n",
            nrow(species_spectra),
            length(unique(species_spectra$Scientific.Names))))
cat(sprintf("  Band columns  : %d\n", length(band_cols)))
cat(sprintf("  Saved RDS     : %s\n", OUT_RDS))
cat(sprintf("  Saved CSV     : %s\n", OUT_CSV))
cat(strrep("=", 55), "\n")
cat("\nNext: run LA_Hyperspectral_Pipeline.ipynb\n")